# Experimento estatístico: 10.000 pares de vértices

Este notebook amplia o estudo pontual para uma amostra de pares de vértices da rede viária. A motivação é que um único trajeto pode ser influenciado por particularidades locais; ao comparar milhares de pares, obtemos uma avaliação mais robusta das métricas $L^p$ como aproximações da distância intrínseca $d_G$.

Para cada par $(u,v)$ e cada métrica $d_p$, calculamos:

$$\tau_p(u,v)=\frac{d_G(u,v)}{d_p(u,v)},$$

além de medidas de erro como MAE, MAPE, RMSE e viés médio.

## 1. Importações

Esta célula importa as bibliotecas gerais e o pacote do projeto. Os notebooks foram separados para deixar o estudo pontual e o estudo estatístico independentes.

In [ ]:
from pathlib import Path
import sys
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import nao_e_so_reta as sm

print("Raiz do projeto:", PROJECT_ROOT)

## 2. Configurações do experimento

Aqui definimos o tamanho da amostra, os valores de $p$, o grafo usado e as pastas de saída. Para testes rápidos, reduza `N_PAIRS` para `500` ou `1000`; para a versão final, use `10_000`.

In [ ]:
CFG = sm.AppConfig()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
for folder in [DATA_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

P_VALUES = sm.DEFAULT_P_VALUES

N_PAIRS = 10000
N_ORIGINS = 250
SEED = 7

# Opções: "barao_recorte", "campinas".
# O recorte de Barão Geraldo costuma ser mais estável no Overpass e ainda está em Campinas.
STATS_MODE = "recorte_ampliado"

# Centro aproximado Barão Geraldo / Unicamp.
STATS_CENTER_POINT = (-22.8190, -47.0700)

# Tamanho do recorte em metros.
# Sugestões:
# 6000  -> Barão Geraldo mais imediato
# 8000  -> Barão Geraldo + entorno próximo
# 10000 -> região maior, ainda bem menor que Campinas inteira
# 12000 -> recorte amplo, pode demorar mais
STATS_DIST_M = 9000

print("Valores de p:", [sm.p_value_label(p) for p in P_VALUES])
print("Pares:", N_PAIRS)
print("Origens distintas:", N_ORIGINS)

## 3. Download ou carregamento do grafo

O grafo é salvo localmente após o primeiro download. Para evitar problemas de conexão com consultas muito grandes, o modo padrão usa um recorte circular em torno de Barão Geraldo. Caso queira testar Campinas inteira, altere `STATS_MODE` para `"campinas"`.

In [ ]:
import osmnx as ox

ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.requests_timeout = 600

if STATS_MODE == "campinas":
    graph_path = DATA_DIR / "graph_campinas.graphml"

    G = sm.load_or_download_graph(
        graph_path,
        place=CFG.place_stats,
        network_type=CFG.network_type,
        make_undirected=CFG.make_undirected,
        keep_largest_component=True,
    )

elif STATS_MODE == "barao_recorte":
    graph_path = DATA_DIR / f"graph_barao_geraldo_{CFG.barao_dist_m}m.graphml"

    G = sm.load_or_download_graph(
        graph_path,
        place=None,
        center_point=CFG.barao_center,
        dist=CFG.barao_dist_m,
        network_type=CFG.network_type,
        make_undirected=CFG.make_undirected,
        keep_largest_component=True,
    )

elif STATS_MODE == "recorte_ampliado":
    graph_path = DATA_DIR / f"graph_barao_entorno_{STATS_DIST_M}m.graphml"

    G = sm.load_or_download_graph(
        graph_path,
        place=None,
        center_point=STATS_CENTER_POINT,
        dist=STATS_DIST_M,
        network_type=CFG.network_type,
        make_undirected=CFG.make_undirected,
        keep_largest_component=True,
    )

else:
    raise ValueError(
        "STATS_MODE deve ser 'barao_recorte', 'recorte_ampliado' ou 'campinas'."
    )

print("Modo:", STATS_MODE)
print("Arquivo:", graph_path)
print(f"Vértices: {len(G.nodes):,}")
print(f"Arestas: {len(G.edges):,}")
print("CRS:", G.graph.get("crs"))

### Visualização da região analisada

A figura abaixo mostra a malha viária do recorte utilizado no experimento estatístico. Ela serve para verificar visualmente se o raio escolhido inclui Barão Geraldo e regiões próximas, sem abranger Campinas inteira.

In [ ]:
fig, ax = ox.plot_graph(
    G,
    node_size=0,
    edge_linewidth=0.6,
    edge_alpha=0.75,
    bgcolor="white",
    show=False,
    close=False,
)

ax.set_title(
    f"Malha viária do recorte analisado — raio de {STATS_DIST_M/1000:.1f} km",
    fontsize=13,
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / f"regiao_recorte_{STATS_DIST_M}m.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 4. Amostragem dos pares de vértices

A amostragem usa várias origens e vários destinos por origem. Isso mantém os pares aleatórios, mas reduz o custo computacional, porque o algoritmo de Dijkstra é executado uma vez por origem distinta em vez de uma vez por par.

In [ ]:
pairs = sm.sample_vertex_pairs(
    G,
    n_pairs=N_PAIRS,
    n_origins=N_ORIGINS,
    seed=SEED,
)

pairs_path = RESULTS_DIR / "sampled_pairs.csv"
sm.save_dataframe(pairs, pairs_path)

print(pairs.head())
print(f"Total de pares: {len(pairs):,}")
print(f"Origens distintas: {pairs['origin'].nunique():,}")
print("Pares salvos em:", pairs_path)

## 5. Cálculo de $d_G$, $d_p$ e $	au_p$

Esta é a etapa principal do experimento. Para cada par de vértices, calculamos a distância intrínseca $d_G$, as distâncias planas $d_p$ e as tortuosidades relativas $	au_p=d_G/d_p$.

In [ ]:
t0 = time.perf_counter()

results = sm.compute_pair_metrics(
    G,
    pairs,
    p_values=P_VALUES,
    show_progress=True,
)

elapsed = time.perf_counter() - t0
results_path = RESULTS_DIR / "results_10000_pairs.csv"
sm.save_dataframe(results, results_path)

print(f"Cálculo concluído em {elapsed/60:.2f} minutos.")
print("Resultados salvos em:", results_path)
print(results.head())

## 6. Busca numérica do melhor $p$

Além dos valores fixos de $p$, avaliamos uma grade fina de valores entre 1 e 5. Essa etapa não recalcula caminhos mínimos; ela reaproveita as coordenadas dos pares e a coluna $d_G$ já calculada.

In [ ]:
P_GRID = sm.p_grid(1.0, 5.0, 0.01)

grid_results = sm.evaluate_p_grid(results, P_GRID)
grid_results_path = RESULTS_DIR / "p_grid_results.csv"
sm.save_dataframe(grid_results, grid_results_path)

best_by_mape = sm.best_p_by(grid_results, criterion="MAPE (%)")
best_by_mae = sm.best_p_by(grid_results, criterion="MAE (m)")

best_p_mape = float(best_by_mape["p"])
best_p_mae = float(best_by_mae["p"])

print("Melhor p por MAPE:")
display(best_by_mape.to_frame().T)

print("Melhor p por MAE:")
display(best_by_mae.to_frame().T)

print("Grade salva em:", grid_results_path)

In [ ]:
from nao_e_so_reta.norms import distance_col, tau_col
from nao_e_so_reta import analysis as an

best_p = float(best_by_mape["p"])

best_distance_col = distance_col(best_p)
best_tau_col = tau_col(best_p)

if best_distance_col not in results.columns:
    results = an.add_metric_columns(
        results,
        p_values=[best_p],
        graph_col="d_graph_m",
    )

print(f"Melhor p segundo MAPE: p* = {best_p:.2f}")
print(f"Coluna da distância: {best_distance_col}")
print(f"Coluna da tortuosidade: {best_tau_col}")

In [ ]:
results = an.add_metric_columns(
    results,
    p_values=[best_p],
    graph_col="d_graph_m",
)

P_VALUES_WITH_BEST = sorted(
    set([*P_VALUES, best_p]),
    key=lambda x: float("inf") if np.isinf(x) else float(x),
)

## 7. Curvas de erro em função de $p$

Esses gráficos mostram como o erro muda conforme variamos $p$. Eles ajudam a discutir se o melhor valor está próximo de uma métrica clássica, como $p=1$, $p=2$, ou se aparece em um valor intermediário.

In [ ]:
mape_fig_path = FIGURES_DIR / "mape_por_p.png"
fig, ax = sm.plot_error_by_p(grid_results, criterion="MAPE (%)", filepath=mape_fig_path)
plt.show()
print("Figura salva em:", mape_fig_path)

mae_fig_path = FIGURES_DIR / "mae_por_p.png"
fig, ax = sm.plot_error_by_p(grid_results, criterion="MAE (m)", filepath=mae_fig_path)
plt.show()
print("Figura salva em:", mae_fig_path)

## 8. Resumo de erros e tortuosidades

Para cada métrica $d_p$, calculamos:

$$MAE=\frac{1}{n}\sum_{i=1}^n |d_p^{(i)}-d_G^{(i)}|,$$

$$MAPE=\frac{100}{n}\sum_{i=1}^n\frac{|d_p^{(i)}-d_G^{(i)}|}{d_G^{(i)}},$$

além de RMSE, viés médio e estatísticas de $\tau_p=d_G/d_p$. A tabela é ordenada pelo MAPE.

In [ ]:
summary = sm.summarize_metric_errors(results, p_values=P_VALUES_WITH_BEST)
summary_path = RESULTS_DIR / "summary_metric_errors.csv"
sm.save_dataframe(summary, summary_path)

display(
    summary[[
        "Métrica", "p", "MAE (m)", "MAPE (%)", "RMSE (m)",
        "Viés médio (m)", "Tortuosidade média", "Tortuosidade mediana"
    ]].style.format({
        "MAE (m)": "{:.1f}",
        "MAPE (%)": "{:.2f}",
        "RMSE (m)": "{:.1f}",
        "Viés médio (m)": "{:.1f}",
        "Tortuosidade média": "{:.3f}",
        "Tortuosidade mediana": "{:.3f}",
    })
)
print("Resumo salvo em:", summary_path)

## 9. Cinco melhores métricas

Nesta etapa selecionamos as cinco melhores métricas segundo o MAPE. Essa seleção será usada no gráfico de dispersão para evitar uma visualização poluída com muitas curvas ao mesmo tempo.

In [ ]:
top5 = sm.top_metrics(summary, n=5, criterion="MAPE (%)")
display(top5[["Métrica", "p", "MAE (m)", "MAPE (%)", "Tortuosidade média", "Tortuosidade mediana"]].style.format({
    "MAE (m)": "{:.1f}",
    "MAPE (%)": "{:.2f}",
    "Tortuosidade média": "{:.3f}",
    "Tortuosidade mediana": "{:.3f}",
}))

## 10. Scatter plot das cinco melhores métricas

O gráfico compara $d_G$ no eixo horizontal com $d_p$ no eixo vertical para as cinco melhores métricas. A linha tracejada é a identidade $y=x$: quanto mais próximos os pontos estão dessa linha, melhor a aproximação direta.

In [ ]:
scatter_path = FIGURES_DIR / "scatter_top5_metricas_vs_dG.png"
fig, ax = sm.plot_top_metric_scatters(
    results,
    summary,
    n=3,
    criterion="MAPE (%)",
    filepath=scatter_path,
    max_points=4000,
)
plt.show()
print("Figura salva em:", scatter_path)

## 11. Tortuosidade relativa por métrica

Agora visualizamos a distribuição das tortuosidades $	au_p=d_G/d_p$ para todas as métricas fixas. A linha horizontal em 1 indica igualdade entre a distância no grafo e a distância plana considerada.

In [ ]:
tau_boxplot_path = FIGURES_DIR / "boxplot_tortuosidade_por_metrica.png"
fig, ax = sm.plot_tortuosity_boxplot(
    results,
    P_VALUES_WITH_BEST,
    filepath=tau_boxplot_path,
)
plt.show()
print("Figura salva em:", tau_boxplot_path)

## 12. Discussão

O estudo estatístico reduz a influência de um único trajeto específico. A comparação por MAE e MAPE indica quais métricas aproximam melhor a distância intrínseca em média, enquanto a tortuosidade $	au_p$ mostra como a relação $d_G/d_p$ se distribui na malha urbana.

Quando $	au_p$ fica próximo de 1, a métrica $d_p$ está próxima da distância no grafo. Valores sistematicamente maiores que 1 indicam que a distância no grafo tende a exceder aquela métrica; valores menores que 1 indicam que a métrica plana tende a superestimar $d_G$.

## 13. Conclusões

Este experimento permite comparar métricas normadas no plano com a distância intrínseca de uma rede real de ruas. A partir dos resultados, é possível discutir não apenas qual métrica apresentou menor erro médio, mas também como a geometria da malha viária afeta a relação entre distância plana e distância na rede.

Para o texto final, recomenda-se destacar: a métrica com menor MAPE, a métrica com menor MAE, a distribuição das tortuosidades e a diferença entre o estudo pontual e o comportamento estatístico.